# InHire API Reverse Engineering Scanner

Este notebook analisa o frontend das páginas InHire para descobrir endpoints de API escondidos dentro dos bundles JS.

Etapas:
1. Baixar HTML da página
2. Detectar bundles JS
3. Baixar JS
4. Extrair URLs
5. Detectar endpoints de API
6. Testar endpoints
7. Gerar relatório


In [ ]:
import requests
import re
import pandas as pd
from urllib.parse import urljoin

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*"
}

## Empresas para análise

In [ ]:
EMPRESAS = [
    "sympla",
    "olist",
    "cora",
    "contabilizei"
]

## Função para extrair JS do HTML

In [ ]:
def extrair_js(html):
    pattern = r'src="(.*?\.js)"'
    return re.findall(pattern, html)

## Função para extrair URLs dentro do JS

In [ ]:
def extrair_urls(js_text):
    regex = r'https://[^"\' ]+'
    return re.findall(regex, js_text)

## Testar endpoint

In [ ]:
def testar_endpoint(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        return {
            "url": url,
            "status": r.status_code,
            "size": len(r.text),
            "preview": r.text[:200]
        }
    except Exception as e:
        return {
            "url": url,
            "status": "erro",
            "size": 0,
            "preview": str(e)
        }

## Scanner principal

In [ ]:
todas_urls = []

for empresa in EMPRESAS:

    base = f"https://{empresa}.inhire.app"
    print("\nEmpresa:", empresa)

    try:

        html = requests.get(base, headers=HEADERS).text

        js_files = extrair_js(html)

        print("JS encontrados:", len(js_files))

        for js in js_files:

            js_url = urljoin(base, js)

            print("Baixando:", js_url)

            try:
                js_text = requests.get(js_url, headers=HEADERS).text

                urls = extrair_urls(js_text)

                for u in urls:

                    todas_urls.append({
                        "empresa": empresa,
                        "url": u
                    })

            except:
                pass

    except:
        print("Erro ao acessar empresa")

## URLs encontradas

In [ ]:
df_urls = pd.DataFrame(todas_urls)

df_urls.drop_duplicates(inplace=True)

df_urls

## Filtrar possíveis APIs

In [ ]:
possiveis_api = df_urls[
    df_urls['url'].str.contains('api|jobs|vagas|graphql', case=False)
]

possiveis_api

## Testar endpoints encontrados

In [ ]:
resultados = []

for url in possiveis_api['url']:

    print("Testando:", url)

    res = testar_endpoint(url)

    resultados.append(res)

## Resultados finais

In [ ]:
df_resultados = pd.DataFrame(resultados)

df_resultados

## Salvar relatório

In [ ]:
df_urls.to_csv("urls_encontradas_js.csv", index=False)

df_resultados.to_csv("teste_endpoints_encontrados.csv", index=False)

print("Relatórios salvos.")